# IFFT signed-green + abs-Gaussian mapping
逐步运行；原始数据不覆盖。每一步的理由详见docs/METHODS_zh.md。

In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import matplotlib.pyplot as plt
ROOT = Path.cwd() if (Path.cwd()/"src").exists() else Path.cwd().parent
sys.path.insert(0,str(ROOT/"src"))
from pipeline import *
F, metadata = load_image(ROOT/"data/input_ifft.tif")
cfg=json.loads((ROOT/"configs/default.json").read_text())
print(metadata)

## 1. 有符号灰度线性映射到绿色
保持正负值次序；全图极值映射0–255，gamma=1。

In [ ]:
L, bounds, clipped = linear_signed_display(F)
green, _, _ = render_layers(L, np.ones(F.shape,dtype=bool))
plt.figure(figsize=(8,8)); plt.imshow(green, interpolation="nearest"); plt.axis("off");
print("Display bounds:",bounds,"clipped:",clipped)

## 2. 原始IFFT取绝对值，再Gaussian sigma=5 px
注意这里用F，不是绿色通道L。此指标仅用于区域选择，不替代signed条纹。

In [ ]:
absolute, smooth = amplitude_map(F, sigma=5)
plt.figure(figsize=(8,8)); plt.imshow(smooth,cmap="magma"); plt.colorbar();

## 3. 审核并设定阈值
Otsu只是起点。修改T后重新运行本单元格及后续单元格；不要把它当作已验证晶界。

In [ ]:
T=otsu_threshold(smooth)
# T=959.9598835876563  # 或者输入审核后的绝对阈值
mask=smooth>=T
plt.figure(figsize=(8,8)); plt.imshow(mask,cmap="gray");
print("T:",T,"selected fraction:",mask.mean())

## 4. 二值alpha只决定显示区域
选区内使用第一步相同RGB，不用平滑图替代条纹。

In [ ]:
rgb, rgba, black = render_layers(L,mask)
plt.figure(figsize=(8,8)); plt.imshow(black,interpolation="nearest"); plt.axis("off");

## 5. 保存新的完整运行记录
修改run名称后运行，避免覆盖旧版本。可另加tem_path，并在cfg中设置TEM显示窗口。

In [ ]:
cfg["threshold"]={"mode":"manual","value":float(T)}
# 取消下一行注释即可保存；已存在且非空的目录会被拒绝覆盖。
# manifest=run_pipeline(ROOT/"data/input_ifft.tif", ROOT/"runs/notebook_run_001",cfg)